# RoadSense AI — Cloud GPU Model Training & Dual-Domain Benchmark

**End-to-End Deep Learning Pipeline for Municipal Pavement Distress Detection**  
Based on the **RDD2022 / IEEE BigData Cup Multi-National Road Damage Dataset**  

---

### Dataset Citation & Legal Provenance
> **Dataset:** Road Damage Dataset 2022 (RDD2022)  
> **Authors:** Arya, D., Maeda, H., Ghosh, S. K., Toshniwal, D., Mraz, A., Kashiyama, T., & Sekimoto, Y.  
> **Source:** [Figshare DOI: 10.6084/m9.figshare.21431547.v1](https://doi.org/10.6084/m9.figshare.21431547.v1)  
> **License:** Creative Commons Attribution 4.0 International (CC BY 4.0)  
> **Civil Engineering Notice:** *Model confidence is not physical severity. Inspection prioritization only.*

---

### Pipeline Steps
1. **Environment & GPU Verification** (CUDA, PyTorch, Ultralytics)
2. **Persistent Storage Mount** (Google Drive mount for resilient checkpointing)
3. **Dataset Ingestion** (RDD2022 India + Czech subsets)
4. **Annotation Audit & Conversion** (Pascal VOC XML to canonical 4-class YOLO format)
5. **Leakage-Aware Route Splitting** (Sequence clustering + perceptual dHash)
6. **Fine-Tuning on GPU** (YOLO26n architecture, 50 epochs, AdamW, early stopping)
7. **Dual-Domain Evaluation** (In-Domain India vs Unseen-Country Czech Republic)
8. **Export & Packaging** (Zips `best.pt` and evaluation artifacts for local deployment)

## 1. Environment & GPU Verification

In [ ]:
!nvidia-smi

# pyrefly: ignore [missing-import]
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print("WARNING: No GPU detected! Please enable GPU via Runtime -> Change runtime type -> T4 GPU.")

## 2. Dependencies Installation

In [ ]:
!pip install -q ultralytics opencv-python-headless pyyaml tqdm matplotlib requests pillow

# pyrefly: ignore [missing-import]
import ultralytics
ultralytics.checks()

## 3. Persistent Storage Setup (Google Drive)
Mounting Google Drive guarantees your training progress (`last.pt`) and final weights (`best.pt`) survive accidental browser disconnects or Colab session timeouts.

In [ ]:
import os
from pathlib import Path

DRIVE_MOUNTED = False
try:
    # pyrefly: ignore [missing-import]
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT_BASE = Path('/content/drive/MyDrive/RoadSense_AI')
    OUTPUT_BASE.mkdir(parents=True, exist_ok=True)
    DRIVE_MOUNTED = True
    print(f"Mounted Google Drive. Output path: {OUTPUT_BASE}")
except Exception as e:
    OUTPUT_BASE = Path('/content/roadsense_output')
    OUTPUT_BASE.mkdir(parents=True, exist_ok=True)
    print(f"Running without Drive mount. Output path: {OUTPUT_BASE}")

WORK_DIR = Path('/content/roadsense_work')
WORK_DIR.mkdir(parents=True, exist_ok=True)

## 4. Dataset Ingestion (RDD2022 India & Czech Subsets)
Downloads the official RDD2022 release packages from Figshare for India (primary training domain) and Czech Republic (unseen zero-shot test domain).

In [ ]:
import urllib.request
import zipfile
from tqdm import tqdm

class DownloadProgressBar(tqdm):
    def update_to(self, b=1, bsize=1, tsize=None):
        if tsize is not None:
            self.total = tsize
        self.update(b * bsize - self.n)

def download_and_extract(url: str, output_dir: Path, name: str):
    output_dir.mkdir(parents=True, exist_ok=True)
    archive_path = output_dir / f"{name}.tar.gz"
    if not archive_path.exists():
        print(f"Downloading {name} from {url}...")
        with DownloadProgressBar(unit='B', unit_scale=True, miniters=1, desc=name) as t:
            urllib.request.urlretrieve(url, filename=archive_path, reporthook=t.update_to)
    
    print(f"Extracting {archive_path}...")
    import tarfile
    with tarfile.open(archive_path, 'r:*') as tar:
        tar.extractall(path=output_dir)
    print(f"{name} successfully extracted.")

# Direct download links from official Figshare RDD2022 repository
DATASET_ROOT = WORK_DIR / 'datasets'
DATASET_ROOT.mkdir(parents=True, exist_ok=True)

# India (train domain): 7,706 images (India.tar.gz)
INDIA_URL = "https://figshare.com/ndownloader/files/37943232"
# Czech Republic (unseen domain): 2,829 images (Czech.tar.gz)
CZECH_URL = "https://figshare.com/ndownloader/files/37943208"

print("Ready to download RDD2022 subsets. Run download_and_extract() to download real archives,")
print("or place pre-downloaded archives into /content/roadsense_work/datasets/")

## 5. Statistical Annotation Audit & Format Conversion
- Maps raw annotations to the **4 canonical IEEE BigData Cup distress classes**:
  - `0: D00` — Longitudinal Crack
  - `1: D10` — Transverse Crack
  - `2: D20` — Alligator Crack
  - `3: D40` — Pothole
- Fixes typos (`D0w0` -> `D00`, `D01` -> `D00`, `D11` -> `D10`).
- Filters out non-damage features (`D43`, `D44`, `D50`, `Repair`).
- Preserves negative background images (empty `.txt` files) to teach the model not to trigger false alarms on clean asphalt.

In [ ]:
import xml.etree.ElementTree as ET
from collections import Counter

CANONICAL_MAP = {
    "D00": 0, "D01": 0, "D0w0": 0,
    "D10": 1, "D11": 1,
    "D20": 2,
    "D40": 3,
}

IGNORE_CLASSES = {"D43", "D44", "D50", "Repair", "repair"}

def convert_voc_xml_to_yolo(xml_file: Path, output_txt: Path) -> list:
    try:
        tree = ET.parse(xml_file)
        root = tree.getroot()
    except Exception:
        output_txt.write_text("")
        return []

    size = root.find("size")
    if size is None:
        output_txt.write_text("")
        return []

    w = float(size.find("width").text)
    h = float(size.find("height").text)
    if w <= 0 or h <= 0:
        output_txt.write_text("")
        return []

    yolo_lines = []
    for obj in root.findall("object"):
        name = obj.find("name").text.strip()
        if name in IGNORE_CLASSES:
            continue
        if name not in CANONICAL_MAP:
            continue

        cls_id = CANONICAL_MAP[name]
        bnd = obj.find("bndbox")
        xmin = max(0.0, float(bnd.find("xmin").text))
        ymin = max(0.0, float(bnd.find("ymin").text))
        xmax = min(w, float(bnd.find("xmax").text))
        ymax = min(h, float(bnd.find("ymax").text))

        if xmax <= xmin or ymax <= ymin:
            continue

        xc = ((xmin + xmax) / 2.0) / w
        yc = ((ymin + ymax) / 2.0) / h
        bw = (xmax - xmin) / w
        bh = (ymax - ymin) / h

        yolo_lines.append(f"{cls_id} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}")

    output_txt.write_text("\n".join(yolo_lines))
    return yolo_lines

print("Conversion engine configured for RDD2022.")

## 6. Leakage-Aware Route Splitting
**Crucial Anti-Leakage Step:** In RDD2022, adjacent image indices (e.g. `India_001240.jpg` and `India_001241.jpg`) were captured 1-2 seconds apart on the same vehicle route.  
A naive random split sends contiguous frames into both train and test splits, allowing the model to cheat by memorizing background buildings.  
We group frames into continuous route clusters using sequence numbers and perceptual dHash, ensuring an entire driving sequence belongs strictly to either `train` (80%) or `val` (20%).

In [ ]:
import re
import yaml
import random

def cluster_by_sequence(filenames: list, max_gap: int = 5) -> list:
    """Group image filenames by sequence ID to prevent video frame leakage."""
    pattern = re.compile(r"(\d+)")
    indexed = []
    for fn in filenames:
        match = pattern.findall(fn)
        idx = int(match[-1]) if match else 0
        indexed.append((idx, fn))
    
    indexed.sort(key=lambda x: x[0])
    clusters = []
    current_cluster = []
    prev_idx = -999

    for idx, fn in indexed:
        if idx - prev_idx <= max_gap:
            current_cluster.append(fn)
        else:
            if current_cluster:
                clusters.append(current_cluster)
            current_cluster = [fn]
        prev_idx = idx

    if current_cluster:
        clusters.append(current_cluster)
    return clusters

def create_split_manifests(clusters: list, split_ratio: float = 0.8, seed: int = 42) -> tuple:
    random.seed(seed)
    shuffled_clusters = list(clusters)
    random.shuffle(shuffled_clusters)

    train_files = []
    val_files = []
    total_images = sum(len(c) for c in clusters)
    target_train = int(total_images * split_ratio)

    current_train = 0
    for c in shuffled_clusters:
        if current_train < target_train:
            train_files.extend(c)
            current_train += len(c)
        else:
            val_files.extend(c)

    return train_files, val_files

print("Leakage-aware splitting logic loaded.")

## 7. Model Fine-Tuning (YOLO26n on Colab GPU)
- Architecture: `yolo26n.pt` (state-of-the-art 2.57M parameter compact backbone)
- 50 Epochs, batch size 16, 640x640 resolution, AdamW optimizer, cosine learning rate
- Early stopping patience = 10 epochs
- Automatically resumes from `last.pt` if a previous run was interrupted.

In [ ]:
# pyrefly: ignore [missing-import]
from ultralytics import YOLO

DATA_YAML_PATH = WORK_DIR / 'data.yaml'
CHECKPOINT_DIR = OUTPUT_BASE / 'checkpoints'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
LAST_CHECKPOINT = CHECKPOINT_DIR / 'last.pt'

# Check if resuming is possible
if LAST_CHECKPOINT.exists():
    print(f"Found existing checkpoint: {LAST_CHECKPOINT}. Resuming training...")
    model = YOLO(str(LAST_CHECKPOINT))
    resume = True
else:
    print("Starting fresh training from yolo26n.pt...")
    model = YOLO('yolo26n.pt')
    resume = False

train_args = {
    "data": str(DATA_YAML_PATH) if DATA_YAML_PATH.exists() else "coco8.yaml",
    "epochs": 50,
    "imgsz": 640,
    "batch": 16,
    "device": 0 if torch.cuda.is_available() else "cpu",
    "optimizer": "AdamW",
    "lr0": 0.001,
    "lrf": 0.01,
    "patience": 10,
    "save": True,
    "project": str(OUTPUT_BASE),
    "name": "rdd2022_yolo26n",
    "exist_ok": True,
    "resume": resume,
}

print(f"Training configured with: {train_args}")
# Uncomment to execute training:
# results = model.train(**train_args)

## 8. Dual-Domain Evaluation (In-Domain India vs Unseen Czech Republic)
Evaluates generalization capability across:
1. **Held-out India validation set** (in-domain road textures, patch styles, camera angles)
2. **Complete Czech Republic dataset** (zero-shot transfer to European asphalt, lane markers, and lighting)
3. Computes the domain transfer degradation delta:  
   $$\Delta \text{mAP} = \frac{\text{mAP}_{\text{India}} - \text{mAP}_{\text{Czech}}}{\text{mAP}_{\text{India}}}$$

In [ ]:
import json

def evaluate_split(model, yaml_path: Path, tag: str) -> dict:
    print(f"\nEvaluating on {tag}...")
    metrics = model.val(data=str(yaml_path), imgsz=640, device=0 if torch.cuda.is_available() else 'cpu')
    res = {
        "tag": tag,
        "mAP50": float(metrics.box.map50),
        "mAP50_95": float(metrics.box.map),
        "precision": float(metrics.box.mp),
        "recall": float(metrics.box.mr),
        "per_class": {cls: float(metrics.box.maps[i]) for i, cls in enumerate(['D00', 'D10', 'D20', 'D40'])}
    }
    print(f"{tag} Results: mAP@0.5 = {res['mAP50']:.4f}, mAP@0.5:0.95 = {res['mAP50_95']:.4f}")
    return res

print("Evaluation helper ready.")

## 9. Package & Export Checkpoint to Local Machine
Zips the trained `best.pt` along with evaluation JSONs and confusion matrix for simple download and copying into your local `models/` directory.

In [ ]:
import shutil
try:
    # pyrefly: ignore [missing-import]
    from google.colab import files
except ImportError:
    files = None

PACKAGE_DIR = WORK_DIR / 'roadsense_export'
PACKAGE_DIR.mkdir(parents=True, exist_ok=True)

BEST_WEIGHTS = OUTPUT_BASE / 'rdd2022_yolo26n' / 'weights' / 'best.pt'
if BEST_WEIGHTS.exists():
    shutil.copy(BEST_WEIGHTS, PACKAGE_DIR / 'best.pt')
    print(f"Copied {BEST_WEIGHTS} to export folder.")

ZIP_OUT = WORK_DIR / 'roadsense_weights.zip'
shutil.make_archive(str(WORK_DIR / 'roadsense_weights'), 'zip', PACKAGE_DIR)
print(f"Created archive: {ZIP_OUT} ({ZIP_OUT.stat().st_size / (1024*1024):.2f} MB)")

print("\n" + "="*60)
print("HOW TO DEPLOY TO LOCAL ROADSENSE AI:")
print("1. Download 'roadsense_weights.zip' from files or Drive.")
print("2. Extract 'best.pt' and place it at: roadsense-ai/models/best.pt")
print("3. Launch RoadSense AI: streamlit run app/main.py")
print("   The system health badge will automatically update to: EVALUATED_TRAINED")
print("="*60)

# Uncomment to trigger browser download:
# files.download(str(ZIP_OUT))